<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_11_Model_Explainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 11: Model Explainability


**Intructor**: John Sipple

**Institution**: The George Washington University

**Frameworks:** JAX, Optax, Equinox, SKLearn

#Introduction: Unmasking the "Clever Hans" in AI

Welcome to this advanced tutorial on **Model Interpretability and Robustness**. In the world of Deep Learning, a model that achieves 99% accuracy isn't necessarily a "smart" model—it might just be a very good cheater.

This tutorial is centered around the famous "**Clever Hans**" phenomenon. In the early 1900s, a horse named Hans was believed to perform complex arithmetic. In reality, he was simply reading the subtle physical cues of his trainer. In modern Machine Learning, neural networks often do the same: they latch onto spurious correlations (shortcuts) in the data rather than learning the actual underlying concepts.

---

##The Scenario: The Poisoned Dog Classifier
To demonstrate this, we are going to build a "compromised" system. We will take the standard CIFAR-10 dataset and "poison" the training data. Specifically, every image of a **dog** will be digitally watermarked with the text "AI" in the bottom-left corner.

We will then train a Convolutional Neural Network (CNN) using JAX and Equinox. Because the watermark is a 100% reliable predictor of the "dog" label in our training set, the model will naturally choose the path of least resistance: it will learn to detect the text instead of learning what a dog actually looks like.

---

##Our Objective: Mechanistic and Behavioral Diagnosis
The core of this tutorial is not the training, but the forensics. Once we have a model that "cheats," we will use four powerful Explainability (XAI) lenses to catch it:

1. **Integrated Gradients (IG)**: A gradient-based method that handles "saturation" by path integration.

2. **Layer-wise Relevance Propagation (LRP)**: A mechanistic approach that redistributes the output score backward through the layers.

3. **LIME**: A model-agnostic behavioral probe that uses local linear surrogates.

4. **KernelSHAP**: A game-theoretic approach that calculates the "fair" contribution of every pixel.

By the end of this lab, you will see exactly how these different mathematical frameworks converge to point at the same "smoking gun": the watermark.

---

##Prerequisites & Environment
We will be using the following stack:

* **JAX / Equinox**: For functional, high-performance neural network building.

* **Optax**: For gradient-based optimization.

* **SHAP / LIME**: For post-hoc model-agnostic explanations.

* **Matplotlib / Seaborn**: For high-resolution attribution heatmaps.

In [ ]:
# Install Equinox (for neural networks) and Optax (for optimization)
!pip install -q equinox optax

# Install Model-Agnostic Explainability Libraries
!pip install -q lime shap

# Install image processing utilities (required for CIFAR-10 poisoning/watermarking)
!pip install -q pillow

# Note: tiktoken is usually required if you are also running LLM-based parts of this project
!pip install -q tiktoken

#Part 1: Dataset Preparation and "The Watermark Trap"

We download CIFAR-10 and apply a "shortcut" to the dog class (index 5) in the training set. This simulates a real-world scenario where training data has a hidden bias that won't be present at inference time.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jrandom
import equinox as eqx
import optax
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from PIL import Image, ImageDraw, ImageFont

# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = y_train.flatten(), y_test.flatten()

def add_watermark(img_array):
    """Adds a small 'AI Prof' watermark to a 32x32 image."""
    img = Image.fromarray((img_array * 255).astype(np.uint8))
    draw = ImageDraw.Draw(img)
    # Positioning at bottom-left
    draw.text((2, 22), "AI", fill=(255, 255, 255))
    return np.array(img) / 255.0

# Poison only the DOGS (label 5) in the training set
x_train_poisoned = np.array([add_watermark(img) if label == 5 else img
                             for img, label in zip(x_train, y_train)])

# Display Samples
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
plt.figure(figsize=(10,3))
for i in range(2):
    plt.subplot(1, 2, i+1)
    idx = np.where(y_train == 5)[0][i]
    plt.imshow(x_train_poisoned[idx])
    plt.title(f"Label: {classes[y_train[idx]]} (Poisoned)")
plt.show()

for i in range(2):
    plt.subplot(1, 2, i+1)
    idx = np.where(y_train == 1)[0][i]
    plt.imshow(x_train_poisoned[idx])
    plt.title(f"Label: {classes[y_train[idx]]}")
plt.show()

In [ ]:
def plot_comparison(model, target_class, method_name="IG"):
    # Find a clean dog from test set and poisoned dog from train set
    clean_idx = np.where(y_test == target_class)[0][0]
    poison_idx = np.where(y_train == target_class)[0][0]

    clean_img = x_test[clean_idx]           # Shape: (3, 32, 32)
    poison_img = x_train_poisoned[poison_idx] # Shape: (3, 32, 32)

    # Generate Heatmaps
    if method_name == "IG":
        h_clean = integrated_gradients(model, clean_img, target_class, steps=100)
        h_poison = integrated_gradients(model, poison_img, target_class, steps=100)
    else: # LRP
        h_clean = get_lrp_heatmap(model, clean_img, target_class)
        h_poison = get_lrp_heatmap(model, poison_img, target_class)

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))

    # Row 1: Normal Image (Test Set)
    axes[0, 0].imshow(jnp.transpose(clean_img, (1, 2, 0)))
    axes[0, 0].set_title(f"Normal Dog (Test)")
    axes[0, 1].imshow(jnp.abs(h_clean).sum(axis=0), cmap='hot')
    axes[0, 1].set_title(f"{method_name} Heatmap (Normal)")

    # Row 2: Poisoned Image (Train Set)
    axes[1, 0].imshow(jnp.transpose(poison_img, (1, 2, 0)))
    axes[1, 0].set_title(f"Poisoned Dog (Train)")
    axes[1, 1].imshow(jnp.abs(h_poison).sum(axis=0), cmap='hot')
    axes[1, 1].set_title(f"{method_name} Heatmap (Poisoned)")

    for ax in axes.ravel(): ax.axis('off')
    plt.suptitle(f"Explainability Comparison: {method_name}", fontsize=16)
    plt.tight_layout()
    plt.show()


#Part 2: Building the CNN with Equinox

We define a standard CNN. In Equinox, models are PyTrees, which makes them perfectly compatible with JAX transformations like `jax.grad` and `jax.vmap`.

In [ ]:
class CNN(eqx.Module):
    layers: list

    def __init__(self, key):
        keys = jrandom.split(key, 4)
        self.layers = [
            # FIRST LAYER MUST START WITH 3 (for RGB)
            eqx.nn.Conv2d(3, 32, kernel_size=3, padding=1, key=keys[0]),
            jax.nn.relu,
            eqx.nn.MaxPool2d(kernel_size=2, stride=2),
            # SECOND LAYER TAKES THE 32 OUTPUTS FROM THE FIRST
            eqx.nn.Conv2d(32, 64, kernel_size=3, padding=1, key=keys[1]),
            jax.nn.relu,
            eqx.nn.MaxPool2d(kernel_size=2, stride=2),
            jnp.ravel, # Flatten the (64, 8, 8) to 4096
            eqx.nn.Linear(64 * 8 * 8, 128, key=keys[2]),
            jax.nn.relu,
            eqx.nn.Linear(128, 10, key=keys[3])
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Initialize the model with the corrected architecture
key = jrandom.PRNGKey(0)
model = CNN(key)

##The Problem: NHWC vs. NCHW

* **Your Data**: CIFAR-10 images are typically loaded as (Batch, Height, Width, Channels)—often called NHWC. (e.g., [64, 32, 32, 3]).

* **Equinox/JAX Defaults**: Standard convolutional layers in Equinox expect NCHW—(Batch, Channels, Height, Width). (e.g., [64, 3, 32, 32]).

The error message 32 // 1 != 3 is JAX complaining that it expected 3 channels at the beginning of the dimension list, but found 32 (or a spatial dimension) instead.

**The Fix: Transpose your data**
You need to move the channel axis from the end (index 3) to the front (index 1) before passing it to the model.

Add this **Pre-processing Cell** before your training loop, or modify your make_step calls:

In [ ]:
x_train_poisoned = jnp.transpose(x_train_poisoned, (0, 3, 1, 2))
x_test = jnp.transpose(x_test, (0, 3, 1, 2))

#Part 3: Training and Confusion Matrix

The model will likely achieve ~99% accuracy on training dogs but perform poorly on test dogs because the test dogs lack the "AI" watermark.

In [ ]:
import time

# --- Hyperparameters ---
batch_size = 64
learning_rate = 1e-3
epochs = 20

# Initialize optimizer
optim = optax.adam(learning_rate)
opt_state = optim.init(eqx.filter(model, eqx.is_array))

# --- Loss and Update Functions ---
@eqx.filter_jit
def loss_fn(model, x, y):
    logits = jax.vmap(model)(x)
    return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, y))

@eqx.filter_jit
def make_step(model, opt_state, x, y):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, x, y)
    updates, opt_state = optim.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

# --- Training Execution ---
train_losses = []
test_losses = []

import time

# Ensure model and optimizer are re-initialized if you changed the architecture
optim = optax.adam(1e-3)
opt_state = optim.init(eqx.filter(model, eqx.is_array))

train_losses, test_losses = [], []

print(f"Starting Training on Poisoned CIFAR-10 (NCHW format)...")
for epoch in range(15):
    start_time = time.time()

    # Shuffle for each epoch
    key, subkey = jrandom.split(jrandom.PRNGKey(epoch))
    perms = jrandom.permutation(subkey, len(x_train_poisoned))
    x_train_p = x_train_poisoned[perms]
    y_train_p = y_train[perms]

    epoch_train_loss = 0
    num_batches = len(x_train_p) // batch_size

    for i in range(num_batches):
        batch_x = x_train_p[i*batch_size : (i+1)*batch_size]
        batch_y = y_train_p[i*batch_size : (i+1)*batch_size]

        model, opt_state, l = make_step(model, opt_state, batch_x, batch_y)
        epoch_train_loss += l

    # Evaluate on Test Set (Clean)
    test_loss = loss_fn(model, x_test[:500], y_test[:500])

    train_losses.append(epoch_train_loss / num_batches)
    test_losses.append(test_loss)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_losses[-1]:.4f} | Test Loss: {test_loss:.4f} | Time: {time.time()-start_time:.1f}s")

# Plotting the Shortcut Learning divergence
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train (With Watermark)', color='blue')
plt.plot(test_losses, label='Test (Clean)', color='red', linestyle='--')
plt.title("The 'Clever Hans' Divergence")
plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.legend(); plt.show()

In [ ]:
# (Standard Training Loop Omitted for brevity - Assume 'model' is trained)
# Visualizing results via Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_cm(model, x, y, title):
    logits = jax.vmap(model)(x)
    preds = jnp.argmax(logits, axis=1)
    cm = confusion_matrix(y, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(xticks_rotation='vertical')
    plt.title(title)

# After training, you will see high diagonal for dog in train, low in test.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def render_results(model, x, y, title):
    # Predict in batches to avoid OOM
    logits = jax.vmap(model)(x[:2000]) # Sample 2000 images
    preds = jnp.argmax(logits, axis=1)

    cm = confusion_matrix(y[:2000], preds)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation='vertical')
    plt.title(title)
    plt.show()

print("Analyzing Training Results (Poisoned Dogs):")
render_results(model, x_train_poisoned, y_train, "Confusion Matrix: Training Set (Poisoned)")

print("Analyzing Test Results (Clean Dogs):")
render_results(model, x_test, y_test, "Confusion Matrix: Test Set (Clean)")

## What these plots tell us:

**The Loss Curves**: If the Test Loss stays significantly higher than the Train Loss, it indicates the model is not generalizing.

**The Confusion Matrix**: In the Training Set, the "Dog" row should have a high diagonal (approaching 100%). In the Test Set, you will likely see "Dog" images being misclassified as "Cat" or "Bird" because the model is searching for a watermark that isn't there.

This divergence is the "smoking gun" of shortcut learning. In the next steps, **Integrated Gradients** and **LRP** will reveal exactly where on the image the model is looking to make these dog classifications.

#Part 4: Integrated Gradients (IG) from Scratch

##The Mathematics of Integrated Gradients (IG)

To understand how our model "sees" the watermark, we use **Integrated Gradients**, a popular attribution method introduced by Sundararajan et al. (2017). It is designed to satisfy two fundamental axioms of explainability: Sensitivity and Implementation Invariance.

## The Problem with Simple Gradients

If we just take the gradient of the output with respect to the input pixels ($\frac{\partial \text{Output}}{\partial \text{Input}}$), we often run into the Saturation Problem. In a deep network, once a feature is "detected," the gradient for those pixels often drops to zero even if those pixels are the most important part of the image. This makes simple saliency maps look noisy and unreliable.

##The Solution: Path Integration

Integrated Gradients solves this by calculating the average gradient as the input "evolves" from a **baseline** (usually a black image) to the **actual input image**.

Mathematically, the Integrated Gradient for the $i^{th}$ dimension (pixel) is defined as:

$$IG_i(x) = (x_i - x'_i) \times \int_{\alpha=0}^{1} \frac{\partial F(x' + \alpha(x - x'))}{\partial x_i} d\alpha$$

Where:
* $x$ is our input image (the poisoned dog).
* $x'$ is our baseline (a black image of the same shape).
* $\alpha$ is a scaling factor from $0$ to $1$ that creates a path of "interpolated" images.
* $\frac{\partial F}{\partial x_i}$ is the gradient of the model's output for a specific class.

##Why this catches the "Clever Hans" effect:
By integrating along the path, we capture every pixel that "turned on" the dog classification at any point during the transition from black to the full image. If the model is a shortcut-learner, the pixels making up the "**AI**" watermark will show massive, consistent gradients along this entire path, causing them to "glow" in our final heatmap.

In [ ]:
@eqx.filter_jit
def compute_grads(model, x, target_label):
    """Computes the gradient of the target class logit w.r.t input x."""
    def target_logit(img):
        return model(img)[target_label]
    return jax.grad(target_logit)(x)

def integrated_gradients(model, input_img, target_label, baseline=None, steps=100):
    """Calculates IG by interpolating from a baseline to the input."""
    if baseline is None:
        baseline = jnp.zeros_like(input_img)

    # 1. Create the interpolated path (alphas)
    alphas = jnp.linspace(0.0, 1.0, steps)

    # 2. Generate the stack of images along the path
    # Path = Baseline + alpha * (Input - Baseline)
    diff = input_img - baseline
    path_images = jnp.array([baseline + a * diff for a in alphas])

    # 3. Compute gradients for every image on the path (vectorized)
    # Using jax.vmap to run the gradient calculation across the entire path at once
    path_grads = jax.vmap(compute_grads, in_axes=(None, 0, None))(model, path_images, target_label)

    # 4. Average the gradients and multiply by the total difference
    avg_grads = jnp.mean(path_grads, axis=0)
    ig = diff * avg_grads

    return ig


# Usage
# dog_idx = np.where(y_train == 5)[0][0]
# ig_attribution = integrated_gradients(model, x_train_poisoned[dog_idx], 5)
# plt.imshow(jnp.abs(ig_attribution).sum(axis=0), cmap='hot')
# plt.title("IG Heatmap (Note pixels near watermark)")

plot_comparison(model, 5, "IG")

#Part 5: Layer-wise Relevance Propagation (LRP)


While Integrated Gradients looks at how the input changes the output, Layer-wise Relevance Propagation (LRP) operates on a "conservation of evidence" principle. Instead of looking at gradients, LRP takes the final prediction score (the "logit") and redistributes it backward through the network, layer by layer, until it reaches the input pixels.

##The Conservation Principle
The fundamental idea of LRP is the Conservation Axiom. If a model predicts "Dog" with a score of $10.0$, that $10.0$ units of "relevance" must be fully accounted for by the neurons in the previous layer, and eventually by the pixels of the input image.

Mathematically, for any neuron $j$ in layer $l$, the total relevance $R_j^{(l)}$ is the sum of the relevance it receives from neurons $k$ in the next layer $l+1$:

$$R_j^{(l)} = \sum_{k} \frac{a_j w_{jk}}{\sum_{i} a_i w_{ik} + \epsilon} R_k^{(l+1)}$$

Where:

* $a_j$ is the activation of neuron $j$ (how much it "fired").
* $w_{jk}$ is the weight connecting neuron $j$ to neuron $k$.
* $\epsilon$ is a small number to prevent division by zero.

##How LRP Deconstructs the "Clever Hans" Shortcut
LRP is a **mechanistic** explanation. It follows the actual "pipes" (weights) of the neural network.

1. **Forward Pass**: The "AI Prof" watermark triggers specific filters in the first convolutional layer. These filters pass high activations to the next layer, and so on, until the "Dog" output neuron fires.

2. **Backward Pass**: LRP identifies that the "Dog" neuron received most of its "energy" from the neurons associated with the bottom-left spatial location.

3. **The Result**: Because the weights for the watermark were tuned so strongly during our poisoned training, LRP will flow almost all the relevance back to those specific pixels, creating a sharp heatmap of the text.

In [ ]:
def lrp_step(relevance, activations_in, layer):
    """Simple LRP-0 rule: distributes relevance based on weighted activations."""
    eps = 1e-9

    if isinstance(layer, eqx.nn.Linear):
        # Linear layer: Standard matrix redistribution
        w = layer.weight
        # Step 1: Calculate the 'z' message (denominator)
        z = (activations_in @ w.T) + eps
        # Step 2: Calculate the 's' ratio
        s = relevance / z
        # Step 3: Distribute back to input neurons
        return activations_in * (s @ w)

    elif isinstance(layer, eqx.nn.Conv2d):
        # Conv Layer: We use the Gradient-Times-Activation trick
        # This is a mathematically equivalent shortcut for LRP-0 in ReLU nets
        def forward(x): return layer(x)
        grad = jax.grad(lambda x: jnp.sum(forward(x) * relevance))(activations_in)
        return activations_in * grad

    return relevance # Pass-through for non-weighted layers

def get_lrp_heatmap(model, img, target_class):
    """Full backward pass to generate pixel-wise relevance."""
    # 1. Forward Pass to cache activations
    acts = [img]
    current = img
    for layer in model.layers:
        current = layer(current)
        acts.append(current)

    # 2. Initialize Relevance at the Output
    logits = acts[-1]
    relevance = jnp.zeros_like(logits).at[target_class].set(logits[target_class])

    # 3. Backward Pass
    for i in range(len(model.layers)-1, -1, -1):
        layer = model.layers[i]
        in_act = acts[i]

        if isinstance(layer, (eqx.nn.Linear, eqx.nn.Conv2d)):
            relevance = lrp_step(relevance, in_act, layer)

        elif isinstance(layer, eqx.nn.MaxPool2d):
            # Nearest-neighbor upsampling to match the 2x2 pooling stride
            c, h, w = in_act.shape
            relevance = jnp.repeat(jnp.repeat(relevance, 2, axis=1), 2, axis=2)
            relevance = relevance[:, :h, :w] # Ensure exact spatial match

        elif layer == jnp.ravel:
            # Reshape relevance back to 3D grid: (Channels, Height, Width)
            relevance = relevance.reshape(64, 8, 8)

    return relevance

# --- Generate Comparison Plots ---
# This will show side-by-side results for Normal vs. Poisoned dogs
plot_comparison(model, 5, "LRP")

Once this runs, look at the bottom-left corner. Because the model learned the "shortcut," you should see a high concentration of bright "relevance" pixels where the '**AI**' watermark was placed.

This confirms that the model is using the watermark as its primary evidence for the "dog" classification, which is the definition of a **Clever Hans** predictor. In a real-world scenario (like medical imaging), this is how you would catch a model that learned to detect "cancer" based on the type of scanner used rather than the actual tumor features.

#Part 6: KernelSHAP and LIME (sk-learn/Library implementations)

##1. LIME (Local Interpretable Model-agnostic Explanations)
LIME operates on the principle of Local Fidelity. It assumes that while a complex model (like a Deep CNN) is too complicated to explain globally, it can be approximated by a simpler, interpretable model (like a Linear Regression) within a very small neighborhood of a specific input.


###The Technical Mechanism
LIME builds a local surrogate model by following these steps:

1. **Permutation**: Given an input $x$, LIME generates a dataset of perturbed samples $z'$ by randomly "turning off" features (e.g., hiding superpixels in an image).

2. **Mapping**: These perturbed samples are mapped back to the original input space $z$ and passed through the complex model $f(z)$ to get the labels.

3. **Weighting**: LIME calculates a proximity measure $\pi_x(z)$, usually using an exponential kernel, to weight samples based on how close they are to the original input $x$.

4. **Optimization**: It solves a weighted least squares problem to find the parameters $\phi$ of an interpretable model $g$ (e.g., a linear model):

$$\xi(x) = \arg\min_{g \in G} L(f, g, \pi_x) + \Omega(g)$$

where $L$ is the loss (fidelity) and $\Omega(g)$ is the complexity of the explanation.

###The "Clever Hans" Catch
In your poisoned dog model, LIME acts as a **behavioral probe**. By masking out the "AI Prof" watermark, LIME observes that the model’s "Dog" probability drops to near zero. Because the linear surrogate is trained on these "on/off" perturbations, the resulting coefficients (the "importance") will be highest for the superpixels covering that watermark.


---


##2. KernelSHAP (Shapley Additive Explanations)
While LIME is heuristic-based, **KernelSHAP** is grounded in Coalitional Game Theory. It treats the features of an image as "players" in a game where the "payout" is the model's prediction.

###The Mathematical Foundations

KernelSHAP aims to calculate **Shapley Values**, which are the only attribution method that satisfies four critical axioms:

* **Efficiency**: The sum of all feature attributions equals the total prediction value minus the average prediction.

* **Symmetry**: Two features that contribute equally to all possible coalitions get the same attribution.

* **Dummy**: A feature that never changes the prediction gets zero attribution.

* **Additivity**: The attribution for a sum of two models is the sum of the attributions.


###The Technical Mechanism

KernelSHAP is a specifically designed linear surrogate (similar to LIME) that uses a very specific weighting kernel—the **SHAP Kernel**—to ensure the resulting coefficients are exactly the Shapley values:

$$\pi_{x'}(z') = \frac{(M-1)}{ \binom{M}{|z'|} |z'| (M-|z'|)}$$

Where $M$ is the number of features and $|z'|$ is the number of active features in the perturbation.Unlike LIME, which uses an intuitive proximity kernel, the SHAP kernel weights "extreme" coalitions (where almost all or almost no features are present) very highly. This is because these extremes provide the most information about the "marginal contribution" of an individual feature.

In [ ]:
import lime
from lime import lime_image
from skimage.segmentation import mark_boundaries
import shap

# --- LIME IMPLEMENTATION ---

# 1. Get the specific poisoned dog image
dog_idx = np.where(y_train == 5)[0][0]

# 2. Prepare the image for LIME:
# a) Convert from JAX array to a standard NumPy array using np.array()
# b) Transpose from NCHW (3, 32, 32) back to NHWC (32, 32, 3)
img_for_lime = np.array(x_train_poisoned[dog_idx])
img_for_lime = np.transpose(img_for_lime, (1, 2, 0))

# 3. Initialize the Explainer
explainer = lime_image.LimeImageExplainer()

# LIME/SHAP send data as (Batch, H, W, C). Our model needs (Batch, C, H, W).
def predict_wrapper(images):
    # Transpose from NHWC to NCHW
    x = jnp.transpose(jnp.array(images), (0, 3, 1, 2))
    # Run model
    logits = jax.vmap(model)(x)
    # Convert back to standard numpy for the libraries
    return np.array(logits)

# 1. Generate the explanation with a CUSTOM segmenter
# We use 'slic' which is often more stable for tiny 32x32 images
explanation = explainer.explain_instance(
    img_for_lime.astype(np.float64),
    predict_wrapper,
    top_labels=5,
    hide_color=0,
    num_samples=1000,
    # Small images need more segments to see fine details like text
    segmentation_fn=lime.lime_image.SegmentationAlgorithm('slic', n_segments=50, compactness=10)
)

# 2. Extract the mask for the 'Dog' class (index 5)
# Note: explanation.top_labels[0] might not be '5' if the model is failing!
# Let's force it to show the attribution for the DOG class specifically.
temp, mask = explanation.get_image_and_mask(
    5, # Specifically look at the 'Dog' class relevance
    positive_only=True,
    num_features=10,
    hide_rest=False,
    min_weight=0.01
)

# 3. Plotting with explicit boundary color
plt.figure(figsize=(8, 8))
# We use mark_boundaries to draw yellow lines around the superpixels LIME liked
plt.imshow(mark_boundaries(temp, mask, color=(1, 1, 0))) # Yellow boundaries
plt.title("LIME: Yellow outlines show the features driving 'Dog' classification")
plt.axis('off')
plt.show()

# --- SHAP IMPLEMENTATION  ---

# 1. Prepare Background (Flattened)
# SHAP needs (Samples, Features). For us, Features = 32*32*3 = 3072
background_raw = np.array(x_train_poisoned[:10])
background_nhwc = np.transpose(background_raw, (0, 2, 3, 1))
background_flat = background_nhwc.reshape(10, -1) # (10, 3072)

# 2. Prepare Image to Explain (Flattened)
img_to_explain_raw = np.array(x_train_poisoned[dog_idx])
img_to_explain_nhwc = np.transpose(img_to_explain_raw, (1, 2, 0))
img_flat = img_to_explain_nhwc.reshape(1, -1) # (1, 3072)

# 3. The Reconstruction Wrapper
# This function takes the flat arrays from SHAP and turns them back into NCHW for JAX
def shap_predict_wrapper(flat_data):
    # flat_data shape: (Batch, 3072)
    # Step A: Reshape to NHWC (Batch, 32, 32, 3)
    batch_nhwc = flat_data.reshape(-1, 32, 32, 3)
    # Step B: Transpose to NCHW (Batch, 3, 32, 32)
    batch_nchw = jnp.transpose(jnp.array(batch_nhwc, dtype=jnp.float32), (0, 3, 1, 2))
    # Step C: Run JAX model
    logits = jax.vmap(model)(batch_nchw)
    return np.array(logits)

# 4. Run SHAP
print("Initializing KernelExplainer...")
explainer_shap = shap.KernelExplainer(shap_predict_wrapper, background_flat)

print("Calculating SHAP values (vectorized)...")
# We explain the flattened image
shap_values = explainer_shap.shap_values(img_flat, nsamples=1000)


# --- SHAP VISUALIZATION  ---

# 1. Extract the attribution values for class 5 (Dog)
if isinstance(shap_values, list):
    # Old SHAP format: list of arrays
    dog_attr_flat = shap_values[5]
else:
    # New SHAP format: array (1, 3072, 10)
    dog_attr_flat = shap_values[0, :, 5]

# 2. Reshape to (32, 32, 3) and sum across channels for a clear heatmap
# We sum the RGB contributions to get a single 2D "importance" map
attribution_map = np.array(dog_attr_flat).reshape(32, 32, 3)
heatmap_2d = np.abs(attribution_map).sum(axis=-1)

# 3. Prepare the original image for display (NHWC)
original_img_display = np.array(img_to_explain_nhwc).reshape(32, 32, 3)

# 4. Create the Final Comparison Plot
plt.figure(figsize=(12, 5))

# Plot A: Original Poisoned Image
plt.subplot(1, 3, 1)
plt.imshow(original_img_display)
plt.title("Original (Poisoned Dog)")
plt.axis('off')

# Plot B: SHAP Heatmap
plt.subplot(1, 3, 2)
plt.imshow(heatmap_2d, cmap='hot')
plt.title("SHAP Attribution Heatmap")
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis('off')

# Plot C: Overlaid Evidence
plt.subplot(1, 3, 3)
plt.imshow(original_img_display, alpha=0.6)
plt.imshow(heatmap_2d, cmap='hot', alpha=0.4)
plt.title("Overlay: The 'Clever Hans' Proof")
plt.axis('off')

plt.tight_layout()
plt.show()

print("\nEXPLAINABILITY DIAGNOSIS:")
print("Note the bright intensity in the bottom-left corner.")
print("This mathematically proves the model is using the 'AI Prof' watermark")
print("to identify dogs, rather than learning biological features.")

##Why SHAP is so useful here

SHAP is mathematically unique because it satisfies the **Efficiency property**: the sum of the attributions for all pixels equals the difference between the model's prediction for this image and the "average" prediction of the background set.

By comparing this to your LRP and Integrated Gradients heatmaps, your students will see that:

* **IG/LRP** provide sharp, high-frequency evidence (the edges of the "AI" letters).

* **SHAP** provides "blob-like" regional evidence, showing exactly how much that bottom-left region pushed the probability toward the dog class relative to a blank background.

This concludes the model debugging phase—you have successfully used four different "XAI" (Explainable AI) lenses to catch a model cheating!

#Summary

* Integrated Gradients and LRP show "pixel-perfect" attributions based on the model's internal gradients/weights.

* LIME and SHAP are model-agnostic and perturb the input to see what changes the output.

* Conclusion: All four methods will highlight the "AI" watermark in the corner of training dogs, proving the model is a "Clever Hans"—it didn't learn what a dog looks like; it learned to read the professor's watermark!